# Experiment 2: answer trajectories and remasking in LLaDA

Rishab, 17 Aug 2026. Everything here runs offline from saved artifacts, no GPU needed.

**What this covers**

1. How the answer evolves across denoising steps, and whether apparent instability is real (Exp 2A)
2. What happens when we unfreeze committed answer tokens and let the model re-predict (Exp 2B)
3. A confidence gate on the intervention, simulated offline from the paired runs

**Two configs appear throughout.** `fast` is the config in `experiment1_gsm8k.ipynb`
(64 steps, single 256-token block, 4 tokens committed per step, 48.1% accuracy).
`std` is LLaDA's recommended setup (256 steps, 32-token blocks, 80.5% accuracy,
matching published numbers). The accuracy gap is the first thing the team needs
to discuss.

## Setup

Unpack `results_only.tar.gz` at the repo root so `artifacts/` sits next to this notebook.

In [ ]:
import gzip, json, glob, statistics
from math import comb
from collections import Counter
import matplotlib.pyplot as plt

PAIRS = "artifacts/experiment2b_gsm8k/pairs/*.json.gz"
pairs = [json.load(gzip.open(p, "rt")) for p in sorted(glob.glob(PAIRS))]
print(f"{len(pairs)} paired runs loaded")

## 1. The answer commits early

Across all 1319 GSM8K problems at the fast config, the correct answer first appears
at **median step 13 of 64**. At that moment the median problem still has **204 of 256
output tokens masked**. 67% of problems that ever reach the correct answer commit it
while more than half the output is still blank.

Concretely, one problem at step 31 (33 steps still to run):

```
1. Calculate the total number of servings Cynthia eats in 60 days:
   - Total servings = 1 serving/day * 60 days = 60 servings.
2. **Determine how many cartons will spend $16.00 on ice cream after 60 days.

Final answer: $16.00
```

Step 2 of the reasoning is incoherent fragment text. The answer is already committed
and already correct. The derivation (60/15 = 4 cartons, 4 x $4 = $16) gets filled in
around an answer that is already fixed.

**Caveat, and it matters.** At the std config the answer arrives at median step 226 of 256.
That is not the model reasoning more carefully. With 32-token blocks the answer span
sits in the last block and cannot be unmasked before step 225, so it arrives 1.5 steps
after it first becomes possible. In both configs the answer commits as early as the
schedule allows. What differs is how much reasoning exists at that moment: 52 tokens
committed at fast config vs 224 at std config.

## 2. Apparent answer instability is a measurement artifact

The obvious way to study this is to extract the answer from each intermediate decode
and count how often it flips. Doing that produces alarming numbers: at the std config,
**77% of ever-correct problems** appear to become correct, then stop being correct.

Almost none of it is real. Flip causes:

| config | total flips | partial_commit | extractor_jump | other |
|---|---|---|---|---|
| fast (n=1319) | 612 | 413 | 142 | 57 |
| std (n=200) | 560 | 442 | 94 | 24 |

`partial_commit` is a number being written out of order: `14` -> `104`, `55` -> `595`,
`72000` -> `720000`. `extractor_jump` is the answer regex latching onto a different
number that was just committed elsewhere in the text.

I read every one of the 81 residual `other` flips by hand. All but one are also
digit insertions, just in the middle of the number rather than at the ends, which
the classifier's prefix/suffix rule missed. The single remaining candidate
(std problem 373, `240000` -> `4`) turns out to be the extractor reading `240,000`
from the reasoning body, then jumping to a `$240,000` figure being written character
by character in the closing sentence.

**Zero genuine regressions across 1519 problems and two configs.**

This follows mechanically: under standard masked-diffusion decoding, committed tokens
are excluded from future prediction, so the model structurally cannot revise. The
contribution is not the claim itself. It is that the naive measurement reports the
opposite, loudly, and anyone who publishes those flip counts without a cause taxonomy
publishes spurious numbers.

In [ ]:
# Paired outcomes across all 1319 problems
def mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, sum(comb(n, k) for k in range(min(b, c) + 1)) / 2**n * 2)

counts = Counter()
for d in pairs:
    a, b = d["outcome_a"], d["outcome_b"]
    if a is None or b is None:
        counts["unparsed"] += 1
    else:
        counts["C->C" if a and b else "C->W" if a else "W->C" if b else "W->W"] += 1

n = len(pairs)
acc_a = sum(1 for d in pairs if d["outcome_a"] is True) / n
acc_b = sum(1 for d in pairs if d["outcome_b"] is True) / n
helped, hurt = counts["W->C"], counts["C->W"]

print(f"normal   {acc_a:.4f}")
print(f"remasked {acc_b:.4f}   ({(acc_b-acc_a)*100:+.2f} pp)")
print(f"intervened on {sum(1 for d in pairs if d['intervened'])} of {n}")
print(f"helped {helped}  hurt {hurt}  McNemar exact p = {mcnemar(helped, hurt):.4f}")

## 3. Remasking: does unfreezing help?

Run A is normal decoding. Run B is identical (same seed, same config) except that
once the model has fully committed a `Final answer: N` span, those number tokens are
reset to `[MASK]` and re-predicted with the rest of the output as context.

**Result: 48.07% -> 49.51%, +1.44 points. 47 helped, 28 hurt, McNemar exact p = 0.037.**

Significant but small. A concrete case: problem 67 commits `590` at step 45, remasking
lets it reconsider, and it produces `595`, the ground truth. The commitment was the
error, not the reasoning.

Two validity checks worth stating: Run A reproduces experiment 1's final answer exactly
on every problem tested, and the intervention only fires once the number's whole region
is mask-free (an earlier version fired on half-written numbers, remasking a `0` that was
about to become `595` — which would have produced a much more exciting and completely
fake result).

In [ ]:
# Commit-time confidence, split by what remasking did
def gate_stat(d):
    iv = d["run_b"].get("intervention") or {}
    v = list((iv.get("commit_time_confidence") or {}).values())
    return min(v) if v else None

groups = {"C->C": [], "W->C": [], "C->W": [], "W->W": []}
for d in pairs:
    a, b = d["outcome_a"], d["outcome_b"]
    if a is None or b is None:
        continue
    k = "C->C" if a and b else "C->W" if a else "W->C" if b else "W->W"
    s = gate_stat(d)
    if s is not None:
        groups[k].append(s)

for k, v in groups.items():
    print(f"{k:6s} n={len(v):4d}  median min-confidence {statistics.median(v):.3f}")

## 4. Confidence gating

The confidence numbers point somewhere useful:

| outcome | n | median min commit-confidence |
|---|---|---|
| stayed correct (C->C) | 438 | 0.819 |
| remasking fixed (W->C) | 47 | 0.569 |
| remasking broke (C->W) | 28 | 0.647 |
| stayed wrong (W->W) | 507 | 0.639 |

Confidently committed answers were already right, so intervening on them can only
do damage. Low-confidence commits are where the value is.

Because every pair records both outcomes, the gate can be simulated offline with no
extra compute: if the gate blocks the intervention the outcome is exactly Run A's,
and if it allows it the outcome is Run B's.

| threshold | accuracy | gain | n intervened | helped | hurt | p |
|---|---|---|---|---|---|---|
| never (baseline) | 48.07% | — | 0 | — | — | — |
| 0.55 | 49.13% | +1.06 | 207 | 21 | 7 | 0.013 |
| 0.65 | 49.51% | +1.44 | 401 | 33 | 14 | 0.008 |
| **0.75** | **49.89%** | **+1.82** | 566 | 42 | 18 | **0.003** |
| 0.85 | 49.66% | +1.59 | 741 | 45 | 24 | 0.015 |
| always | 49.51% | +1.44 | 1020 | 47 | 28 | 0.037 |

Gating at 0.75 gets a larger gain than always intervening while firing on 45% fewer
problems, and the p-value improves by an order of magnitude.

**Held-out check, because picking a threshold on the same data you evaluate it on is
selection on test data.** Tuning on even problem ids picks 0.70 (+2.12 pp there);
applied to the odd half it gives +0.91 pp (49.47% -> 50.38%, 19 helped, 13 hurt,
p = 0.377). So the effect direction holds out but the magnitude shrinks and loses
significance at n=660. That is the honest number to quote, not +1.82.

In [ ]:
# Reproduce the sweep
def evaluate(thr, subset, always=False):
    correct = ivn = helped = hurt = 0
    for d in subset:
        s = gate_stat(d)
        use_b = (s is not None) and (always or s < thr)
        out = d["outcome_b"] if use_b else d["outcome_a"]
        if use_b:
            ivn += 1
            helped += d["outcome_a"] is False and d["outcome_b"] is True
            hurt += d["outcome_a"] is True and d["outcome_b"] is False
        correct += out is True
    return correct / len(subset), ivn, helped, hurt

base = sum(1 for d in pairs if d["outcome_a"] is True) / len(pairs)
xs, accs, ns = [], [], []
for i in range(21):
    t = round(i * 0.05, 2)
    acc, ivn, _, _ = evaluate(t, pairs)
    xs.append(t); accs.append(acc * 100); ns.append(ivn)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(xs, accs, marker="o", label="accuracy")
ax.axhline(base * 100, ls="--", c="grey", label="no intervention")
ax.set_xlabel("confidence threshold (intervene below)")
ax.set_ylabel("accuracy (%)")
ax2 = ax.twinx()
ax2.plot(xs, ns, c="tab:orange", alpha=0.5, label="n intervened")
ax2.set_ylabel("problems intervened on")
ax.legend(loc="lower right")
plt.title("Confidence-gated remasking, GSM8K n=1319, fast config")
plt.tight_layout()
plt.show()

## 5. Where this leaves us

**What holds up**

- Apparent answer instability in diffusion LMs is essentially all measurement artifact.
  Replicated at two configs, every residual case inspected by hand. This one protects
  the whole group: any experiment-1 write-up that reports flip counts without a cause
  taxonomy would be reporting noise.
- Remasking produces a real if small accuracy gain, +1.44 pp ungated (p = 0.037),
  and confidence gating improves it to +1.82 pp in-sample, +0.91 pp held out.
- The fast config costs 32 accuracy points versus the standard config (48.1% vs 80.5%).

**What does not hold up yet**

- "The answer commits before the reasoning" is specific to single-block decoding. At
  the std config the answer arrives late because the block schedule forces it to.
  Both configs commit as early as permitted; only the amount of surrounding reasoning
  differs. Stating it any more strongly than that invites a one-sentence rejection.
- Everything above is at the 48% config. Nobody deploys that. The remasking result
  needs to hold at the std config to be worth a main-track claim.

**Next, in priority order**

1. Confidence-gated remasking at the std config, a few hundred problems. This is the
   experiment that decides whether we have a method or a probe.
2. A compute-matched baseline. Remasking adds forward passes, so the honest comparison
   is against simply running more denoising steps. Without this, "the gain is just
   extra compute" is the first reviewer objection.
3. MATH500 (Mukarramah has the extractor for boxed answers).
4. Fix the flip classifier: replace prefix/suffix matching with a subsequence check so
   mid-number insertions land in `partial_commit` instead of `other`. Roughly 80 flips
   move buckets; the conclusion does not change.

**For the team to decide:** was 64 steps with a single 256-token block deliberate? It
costs 32 accuracy points against LLaDA's published setup, and it changes the qualitative
picture of when answers emerge. If the aggressive schedule is the point, that is a fine
paper about the cost of fast decoding. If not, the main results should move to std config.